# Day 4 — ILT 2: Building the Bronze Layer — Design & Strategy
### GlobalMart Data Engineering · 1:30 PM – 3:00 PM

---

## Session Objectives

By the end of this session you will be able to:
- Explain the role and responsibilities of the Bronze layer in medallion architecture
- Apply the 3 Golden Rules of Bronze design
- Define the standard audit columns every Bronze table must have
- Design Bronze table schemas for both of GlobalMart's ingestion pathways
- Choose the correct write mode (append vs upsert vs overwrite) per pathway
- Describe how GlobalMart's real Lakeflow Connect pipeline lands data in Bronze — upserted latest state, not an event log

---

## Agenda

| Time | Topic |
|------|-------|
| 1:30 | Bronze in the medallion architecture |
| 1:40 | The 3 Golden Rules of Bronze |
| 1:50 | Audit columns — what, why, which |
| 2:05 | Table naming conventions |
| 2:15 | Write modes — append vs upsert vs overwrite |
| 2:25 | Lakeflow Connect Bronze design |
| 2:40 | GlobalMart's 6 Bronze table designs |
| 2:55 | What NOT to do in Bronze + Q&A |

---
## 1. Bronze in the Medallion Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                  MEDALLION ARCHITECTURE                       │
│                                                               │
│  Sources → BRONZE → SILVER → GOLD → Consumers                │
│                                                               │
│  BRONZE   = Raw ingestion layer                               │
│             "Land first, ask questions later"                 │
│                                                               │
│  SILVER   = Cleaned, conformed, enriched                      │
│             "Make it trustworthy"                             │
│                                                               │
│  GOLD     = Business-ready aggregates                         │
│             "Make it useful" — this is where fact_sales lives │
└──────────────────────────────────────────────────────────────┘
```

### Bronze Is NOT a Staging Area

Common misconception: Bronze = temporary holding area, delete after Silver is built.

**Wrong.** Bronze is permanent.

| Staging Area | Bronze Layer |
|-------------|-------------|
| Temporary — deleted after processing | Permanent — kept forever |
| No audit trail | Full audit trail (when + from where) |
| No time travel | Delta time travel across all historical loads |
| No schema enforcement | Evolves with the source, tracked by Auto Loader / Lakeflow Connect |
| Can't reprocess | Can always reprocess Silver from Bronze |

### Why Keep Bronze Forever?

> **Bronze is your insurance policy.**
>
> If Silver has a bug (wrong transformation, bad join), you don't go back to the source.
> You reprocess from Bronze — faster, cheaper, and the source may have already changed.

Real example:
```
Silver job has a timezone bug → orders from 11 PM to 1 AM assigned wrong date
Fix: correct the transformation, rerun Silver from Bronze → problem solved
Without Bronze: would need to re-extract from Postgres and re-request ADLS file
                history — painful, and the files may no longer be there
```

---
## 2. The 3 Golden Rules of Bronze

### Rule 1 — Land Raw, Transform Never

```
Bronze DOES:                          Bronze DOES NOT:
  ✅ Land incoming data                 ❌ Rename columns
  ✅ Add audit columns                  ❌ Cast data types
  ✅ Preserve source column names       ❌ Join with other tables
  ✅ Handle schema evolution            ❌ Filter rows
  ✅ Write to Delta                     ❌ Aggregate
                                        ❌ Apply business rules
```

The only columns Bronze adds are **audit columns** — everything from the source lands as-is.

---

### Rule 2 — Idempotent Writes

**Idempotent** = running the ingestion multiple times produces the same result. No duplicates.

```
Run 1: products.csv → 12,000 rows in Bronze
Run 2: products.csv → same 12,000 rows (not re-processed)
Run 3: products_v2.csv → 12,040 rows (only 40 new rows added)
```

How we achieve this per pathway:
- **ADLS Autoloader (products, customers, address, payments):** checkpoint — tracks every processed file
- **Postgres Lakeflow Connect (orders, order_items):** cursor's last-seen value — Lakeflow Connect re-queries `WHERE updated_at > last_seen_value` each run and only touches rows that actually changed

---

### Rule 3 — Bronze Never Manually Deletes a Row

```
Both GlobalMart pathways honor this, but differently:

  ✅ Autoloader (products, customers, address, payments): literally append-only —
     every file's rows land as new rows, nothing is ever overwritten or removed
  ✅ Lakeflow Connect (orders, order_items): upserts the latest row per key — but
     still never issues a manual DELETE. If a source row is hard-deleted in Postgres,
     Bronze does NOT remove it either — it simply stops being updated and goes stale.
     Nothing tells the pipeline to delete it, and nothing does.

Neither pathway ever overwrites Bronze wholesale. There is no "full snapshot, overwrite
daily" source in GlobalMart's real pipeline.
```

---
## 3. Audit Columns — The Bronze Standard

Every Autoloader Bronze table must have these audit columns added during ingestion. This matches the real audit-column set used in GlobalMart's actual Bronze tables — deliberately minimal, not a kitchen-sink of "just in case" columns:

| Column | Data Type | Value | Purpose | Which Pathway |
|--------|-----------|-------|----------|----------------|
| `_ingested_at` | timestamp | `current_timestamp()` | When the row was written to Bronze | Autoloader |
| `_source_file` | string | `col("_metadata.file_path")` | Which file this row came from | Autoloader only |

Lakeflow Connect's Bronze tables (`orders`, `order_items`) don't need manual audit columns at all — Databricks manages the governed Unity Catalog table (lineage, history, access) for you, and there's no `_source_file` equivalent to add since there's no file involved.

---

### The Real Pattern

```python
from pyspark.sql.functions import current_timestamp, col

# Autoloader (ADLS file) sources — real pattern from <your-catalog>.bronze.customers:
df = (
    df.withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
)

# Lakeflow Connect (Postgres) sources — no notebook code at all in production.
# Databricks manages the connection, cursor tracking, and the write itself.
```

### Why Keep It This Minimal?

A common instinct is to add `_source_system`, `_batch_id`, `_load_type`, and similar "just in case" columns to every Bronze table. GlobalMart's real build doesn't — and the reason matters:

```
_source_file already answers "which pathway, which file, when" for Autoloader
  (the path itself encodes the source folder — no separate _source_system needed)

Lakeflow Connect needs no manual audit columns at all — Databricks already
governs lineage and history for the managed table

Every extra column is one more thing to keep correct across every ingestion
script, forever. Add a column only when a real query will filter or group by it.
```

---
## 4. Table Naming Conventions

### Unity Catalog Naming — the real, live pattern

GlobalMart runs on Unity Catalog *today* — this isn't a "later" preview, it's how every table you build from here on is named:

```sql
-- Fully qualified: CATALOG.SCHEMA.TABLE
<your-catalog>.bronze.orders
<your-catalog>.bronze.order_items
<your-catalog>.bronze.products
<your-catalog>.bronze.customers
<your-catalog>.bronze.addresses
<your-catalog>.bronze.payments

<your-catalog>.silver.orders          -- cleaned, conformed
<your-catalog>.gold.fact_sales         -- business-ready

-- Catalog = your catalog     (the whole GlobalMart project, your own catalog)
-- Schema  = bronze/silver/gold   (the medallion layer — this IS the namespace,
--                                  so it never needs repeating in the table name)
-- Table   = the entity, plain and simple (no source-system prefix needed —
--                                          the catalog already tells you that)
```

### Why the Table Name Doesn't Need a Prefix

A common instinct coming from path-based storage (`bronze_supabase_orders`, `bronze_adls_products`) is to encode the layer and source into the table name itself. In Unity Catalog, don't — the three-level namespace already carries that information:

```
❌ Redundant:  <your-catalog>.bronze.bronze_adls_products
✅ Clean:      <your-catalog>.bronze.products

The schema "bronze" already says what layer this is.
The catalog already says what project this is.
The table name just needs to say what the data IS: products.
```

### Reading and Writing with Three-Level Names

```python
# No abfss:// path needed for a Unity Catalog managed table — just the name:
df = spark.table(f"{CATALOG}.bronze.products")

df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.bronze.products")

# Or with streaming:
query.toTable(f"{CATALOG}.bronze.products")
```

The external location is only needed for the **source-side** read (the raw CSVs in `raw-data/`) — once the Bronze table itself is created, it's a normal managed table addressed by name, same as any SQL table you've used before.

---
## 5. Write Modes — Append vs Upsert vs Overwrite

### Append (Autoloader's Default — 4 of GlobalMart's 6 Bronze Tables)

Use **append** when the source produces a new batch of rows with no need to reconcile against what's already there:

```
ADLS Auto Loader:   new files land → append rows from each file
```

```python
# Batch append:
df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.bronze.products")

# Streaming append:
raw_stream.writeStream.format("delta").outputMode("append").toTable(f"{CATALOG}.bronze.products")
```

### Upsert (Lakeflow Connect — `orders`, `order_items`)

GlobalMart's real Lakeflow Connect pipeline does **not** append — it **upserts**. Each run re-queries the cursor column and merges the result into Bronze: new keys are inserted, existing keys are updated in place, matching the pipeline's "history tracking: Off (SCD1)" setting. Databricks manages this write for you — there's no notebook code to write for it in production.

```
Run 1: orders_data_ingestion_cdc → full snapshot, ~126,000 rows upserted
Run 2: 40 orders changed since last_seen_value → those 40 rows updated in place,
       Bronze row count stays the same (no new keys) or grows only by however
       many are genuinely new orders
```

### When Overwrite Would Apply (Not Currently Used by GlobalMart)

Overwrite is only correct when a source returns a **full state snapshot with no way to identify just the changed rows** — for example, a small nightly export that always contains 100% of current rows. GlobalMart has no such source today. Know the pattern anyway — you'll meet full-snapshot sources on other projects.

### The Danger of Overwrite on an Append Source

```
❌ Wrong: overwriting Bronze products every day
   Day 1: 12,000 rows → Bronze has 12,000 rows
   Day 2: 12,040 rows → Bronze overwritten → 12,040 rows
   Day 3: discovers Day 1 data had a bug → CANNOT go back

✅ Correct: appending every day
   Day 1: 12,000 rows appended
   Day 2: 40 new rows appended
   Day 3: use time travel to go back to the Day 1 state
```

| Table | Write Mode | Reason |
|-------|------------|--------|
| `orders` | Upsert | Lakeflow Connect re-queries the cursor and upserts changed rows |
| `order_items` | Upsert | Same — Lakeflow Connect, cursor-based |
| `products` | Append | Each file drop = new batch of rows |
| `customers` | Append | Each file drop = new batch of rows |
| `addresses` | Append | Each file drop = new batch of rows |
| `payments` | Append | Each file drop = new batch of rows |

---
## 6. Lakeflow Connect Bronze Design

GlobalMart's real Lakeflow Connect pipeline (query/cursor-based, history tracking Off) lands the **current state**, not an event log — this is a different shape than log-based CDC, and it's worth being precise about which one you're building.

### What Bronze Actually Looks Like

```
<your-catalog>.bronze.orders (upserted, one row per order_id):

order_id   | customer_id | order_date | order_channel | ...       | (no _cdc_op column)
-----------|-------------|------------|----------------|-----------
ORD-001    | CUST-001    | 2026-06-15 | Shipped        | ...
ORD-002    | CUST-002    | 2026-06-15 | Delivered      | ...
ORD-003    | CUST-003    | 2026-06-15 | Pending        | ...

When ORD-001's status later changes Shipped → Delivered:
  the SAME row is updated in place — Bronze still has exactly one row for ORD-001,
  now showing Delivered. There is no second row, no history of the Shipped state.
```

### Key Point: Bronze Holds Latest State Only

```
This is NOT an event log — there's no MERGE happening in Silver to collapse
multiple rows down to one, because Bronze never had multiple rows for the
same order_id to begin with. Lakeflow Connect already did that upsert for you.

For ORD-001:
  Bronze has exactly 1 row, always — whatever its current status is.
  Silver reads this upserted state and applies its own transformations
  (casting, renaming, quality checks) — no MERGE-to-collapse-duplicates step
  needed for this pathway, unlike a genuine append-only event log would need.
```

### The One Real Gap: Hard Deletes

```
If an order is hard-deleted in Postgres:
  → It stops matching the cursor query (WHERE updated_at > last_seen_value)
  → Lakeflow Connect never sees a "this row is gone" event — there isn't one
  → The row stays in Bronze exactly as it last was, silently going stale
  → No downstream layer is told about the deletion

This is the real, named limitation of query/cursor-based capture — see Day 2
for the full explanation and what a log-based (WAL) connector would do differently.
```

### Why This Still Beats a Hand-Rolled Watermark Script

1. **No manual scheduling code** — Databricks manages the run
2. **Governed table** — Unity Catalog lineage, access control, audit out of the box
3. **Less to maintain** — one pipeline definition instead of a JDBC script per table
4. **Same blind spot either way** — hand-rolled watermark and Lakeflow Connect's cursor mode both miss hard deletes; the difference is who writes and maintains the code

In [ ]:
# ─── ILLUSTRATIVE: What Lakeflow Connect does under the hood ──────────────────
# Reference snippet — in production Lakeflow Connect does this automatically,
# no notebook code needed. Shown here so the mechanics aren't a black box:
# an upsert (MERGE) into Bronze on the cursor column, not an append.

"""
from pyspark.sql.functions import col

# Conceptually, each Lakeflow Connect run does something like this MERGE —
# Databricks runs the real equivalent internally, you never write this code:

changed_rows = spark.read.jdbc(url, "(SELECT * FROM orders WHERE updated_at > :last_seen) t", ...)

(DeltaTable.forName(spark, f"{CATALOG}.bronze.orders")
    .alias("target")
    .merge(changed_rows.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
"""

print("Key patterns in Lakeflow Connect's Bronze write (managed, not written by you):")
print("  1. Re-query the cursor column: WHERE updated_at > last_seen_value")
print("  2. Upsert (MERGE) into Bronze — new keys inserted, existing keys updated")
print("  3. History tracking: Off (SCD1) — one row per key, latest state only")
print("  4. .saveAsTable(f'{CATALOG}.bronze.<table>') — Unity Catalog managed table,")
print("     no abfss:// path and no storage key needed")

---
## 7. GlobalMart's 6 Bronze Table Designs

### Lakeflow Connect Pathway — `<your-catalog>.bronze.*`

```
<your-catalog>.bronze.orders:  (upserted — no manual audit columns to add)
  order_id         STRING
  customer_id      STRING
  order_date       STRING    ← kept as STRING in Bronze (Silver casts to DATE)
  shipping_date    STRING
  order_channel    STRING
  ... (all source columns — Databricks may add its own internal metadata,
       but there's nothing for you to write with withColumn())

<your-catalog>.bronze.order_items:  (upserted — no manual audit columns to add)
  order_item_id    STRING
  order_id         STRING
  product_id       STRING
  quantity         STRING
  unit_price       STRING
  ... (all source columns)
```

### Autoloader Pathway (ADLS File Drops) — `<your-catalog>.bronze.*`

```
<your-catalog>.bronze.products:
  _ingested_at     TIMESTAMP
  _source_file     STRING    ← full ADLS path of the source CSV
  product_id       STRING
  product_name     STRING
  category         STRING
  actual_price_inr DOUBLE
  ... (all CSV columns)

<your-catalog>.bronze.customers:
  _ingested_at, _source_file  ← same audit columns
  customer_id      STRING
  first_name       STRING
  last_name        STRING
  email            STRING
  ... (all CSV columns — same shape you explored in Day 1 HOL)

<your-catalog>.bronze.addresses / <your-catalog>.bronze.payments: same audit-column pattern,
  entity-specific business columns (address_id/customer_id/city/... and
  payment_id/order_id/payment_method_id/amount/...)
```

---
## 8. What NOT to Do in Bronze

| Anti-Pattern | Why It's Wrong | Correct Approach |
|-------------|---------------|------------------|
| Casting `order_date` STRING → DATE | If casting fails, you lose data | Cast in Silver |
| Renaming `order_channel` to `channel` | Breaks traceability to source | Rename in Silver |
| Filtering cancelled orders | Might need them later for analysis | Filter in Silver |
| Joining `orders` to `customers` | Bronze tables are independent | Join in Silver/Gold |
| Aggregating daily totals | Aggregation belongs in Gold | Aggregate in Gold |
| Deduplicating Autoloader files | Dedup logic may change | Dedup in Silver |
| Dropping columns that seem unused | Nothing is unused until proved otherwise | Keep all columns |

---

## Key Takeaways

1. **Bronze = permanent raw layer** — not a staging area, kept forever
2. **3 Golden Rules:** land raw, idempotent writes, never manually delete a row — even Lakeflow Connect's upserted tables just go stale, never explicitly delete
3. **Audit columns, kept minimal:** `_ingested_at` + `_source_file` on Autoloader tables — Lakeflow Connect tables need none, Databricks governs them for you
4. **Lakeflow Connect Bronze = upserted latest state** — one row per key, no event log, no `_cdc_op` column, and no capture of hard deletes
5. **Write mode:** append for Autoloader, upsert for Lakeflow Connect — never overwrite
6. **Never transform in Bronze** — land first, transform in Silver
7. **Naming:** `<your-catalog>.<layer>.<entity>` — the catalog/schema already say project and layer, so the table name is just the entity, nothing more

---

## Discussion Questions

1. *An order is hard-deleted in Postgres. Does Bronze `orders` reflect that? Why or why not — and what would have to change for it to?*

2. *Your `<your-catalog>.bronze.products` table has 2 million rows. You discover a bug in the ingestion job that corrupted rows from last Tuesday's file. Without a `_batch_id` column, how would you find and fix the affected rows using `_source_file` and `_ingested_at` instead?*

3. *A junior engineer suggests casting all date columns to `DATE` type in Bronze to make Silver simpler. What's the risk?*

4. *Why does GlobalMart's Autoloader pathway never use overwrite mode, but its Lakeflow Connect pathway effectively overwrites each row in place?*

5. *What happens to `<your-catalog>.bronze.customers` if next week's `customers.csv` has a new column? What config prevents a pipeline crash?*